In [1]:
#!/usr/bin/env python
# coding: utf-8

# # Test Load CSV Files Template
# 
# This notebook tests the `run_from_json` function in `load_csv_files_with_config.py`

# ## 1. Setup and Import

import os
import json
import shutil
import pandas as pd
from pathlib import Path
import sys
import traceback

# Create test directories
os.makedirs('test_run', exist_ok=True)
os.chdir('test_run')
print(f"Working directory: {os.getcwd()}")

# ## 2. Create Test Data
# 
# Create sample CSV files to mimic your actual data

# Create a simplified version of your mcmicro data
data1 = pd.DataFrame({
    'CellID': [1, 2, 3, 4, 5],
    'CD45': [10.5, 20.3, 15.7, 8.2, 12.4],
    'CD3D': [5.2, 8.1, 3.4, 12.3, 6.7],
    'X_centroid': [100, 150, 200, 250, 300],
    'Y_centroid': [100, 120, 140, 160, 180],
    'broad_cell_type': ['T cell', 'B cell', 'T cell', 'Macrophage', 'B cell']
})

data2 = pd.DataFrame({
    'CellID': [6, 7, 8, 9, 10],
    'CD45': [11.5, 22.3, 13.7, 9.2, 14.4],
    'CD3D': [6.2, 7.1, 4.4, 11.3, 5.7],
    'X_centroid': [350, 400, 450, 500, 550],
    'Y_centroid': [200, 220, 240, 260, 280],
    'broad_cell_type': ['T cell', 'NK cell', 'B cell', 'Macrophage', 'T cell']
})

# Save CSV files
os.makedirs('csv_input_dir', exist_ok=True)
data1.to_csv('csv_input_dir/sample_data_1.csv', index=False)
data2.to_csv('csv_input_dir/sample_data_2.csv', index=False)
print("Created test CSV files:")
print(os.listdir('csv_input_dir'))

# ## 3. Create Configuration Table

config_data = pd.DataFrame({
    'file_name': ['sample_data_1.csv', 'sample_data_2.csv'],
    'slide_number': ['Slide_001', 'Slide_002'],
    'patient_id': ['Patient_A', 'Patient_A'],
    'timepoint': ['baseline', 'week4']
})

config_data.to_csv('config_table.csv', index=False)
print("\nConfiguration table:")
print(config_data)

# ## 4. Copy the Template File
# 
# **ACTION NEEDED:** Copy your `load_csv_files_with_config.py` template to this directory

template_path = 'load_csv_files_with_config_template.py'
print(f"\n⚠️  Please copy your template file to: {os.path.abspath(template_path)}")
print("Once copied, continue with the next cell...")

# ## 5. Create Parameters JSON

# Create output directory first
os.makedirs('dataframe_folder', exist_ok=True)

params = {
    "CSV_Files": "csv_input_dir",
    "CSV_Files_Configuration": "config_table.csv",
    "Output_File": "dataframe_folder/combined_data.csv",  # Full path with filename
    "String_Columns": ["broad_cell_type", "slide_number", "patient_id", "timepoint"]
}

# Alternative parameter names that might be expected
# Some templates use different parameter names for output
alternative_params = {
    "CSV_Files": "csv_input_dir",
    "CSV_Files_Configuration": "config_table.csv",
    "DataFrames": "dataframe_folder",  # Directory for output
    "Output_File": "combined_data.csv",  # Just filename
    "String_Columns": ["broad_cell_type", "slide_number", "patient_id", "timepoint"]
}

# Save both versions
with open('params.json', 'w') as f:
    json.dump(params, f, indent=2)
    
with open('params_alt.json', 'w') as f:
    json.dump(alternative_params, f, indent=2)

print("Parameters JSON created (main version):")
print(json.dumps(params, indent=2))
print("\nAlternative parameters (if first doesn't work):")
print(json.dumps(alternative_params, indent=2))

# ## 6. Test the Template Function
# 
# This simulates what Galaxy would do

def test_template_execution():
    """Test the template's run_from_json function"""
    
    # Ensure output directory exists
    os.makedirs('dataframe_folder', exist_ok=True)
    
    try:
        # Check if template exists
        if not os.path.exists(template_path):
            print(f"❌ Template not found at: {template_path}")
            print("Please copy the template file and try again.")
            return
        
        # Import the template
        import importlib.util
        spec = importlib.util.spec_from_file_location("template_mod", template_path)
        if not spec or not spec.loader:
            print("❌ Could not load module spec")
            return
        
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        print("✓ Template module loaded successfully")
        
        # Check what functions/classes are available in the module
        print("\n📋 Available in template module:")
        for item in dir(mod):
            if not item.startswith('_'):
                print(f"   - {item}")
        
        # Check for run_from_json function
        if not hasattr(mod, 'run_from_json'):
            print('❌ Template missing run_from_json function')
            print('   Looking for alternative entry points...')
            
            # Check for class-based approach
            if hasattr(mod, 'LoadCSVFilesWithConfig'):
                print('   Found LoadCSVFilesWithConfig class')
                # You might need to instantiate and call it differently
            return
        
        # Execute the template
        print("\n📊 Executing template with params.json...")
        result = mod.run_from_json('params.json', save_results=True, show_plot=False)
        print(f"✓ Template executed successfully!")
        print(f"   Result type: {type(result).__name__}")
        
        # Check outputs
        print("\n📁 Checking outputs:")
        
        # Check dataframe_folder
        if os.path.exists('dataframe_folder'):
            files = os.listdir('dataframe_folder')
            print(f"   Files in dataframe_folder: {files}")
            
            # Check for the combined file
            output_files = ['combined_data.csv', 'combined_dataframe.csv', 'output.csv']
            found_output = False
            
            for output_name in output_files:
                if output_name in files:
                    output_path = f'dataframe_folder/{output_name}'
                    df = pd.read_csv(output_path)
                    print(f"\n✓ Output file '{output_name}' created successfully!")
                    print(f"   Shape: {df.shape}")
                    print(f"   Columns: {list(df.columns)}")
                    print(f"\n   First few rows:")
                    print(df.head())
                    
                    # Check if configuration columns were added
                    if 'slide_number' in df.columns:
                        print(f"\n✓ Configuration columns successfully added!")
                        print(f"   Unique slide numbers: {df['slide_number'].unique()}")
                        print(f"   Unique patients: {df['patient_id'].unique()}")
                        print(f"   Unique timepoints: {df['timepoint'].unique()}")
                    else:
                        print("\n⚠️  Configuration columns not found in output")
                    
                    found_output = True
                    break
            
            if not found_output:
                print("\n⚠️  Expected output file not found")
                print(f"   Available files: {files}")
                
                # Check if there's a pickle file
                pickle_files = [f for f in files if f.endswith('.pickle') or f.endswith('.pkl')]
                if pickle_files:
                    print(f"\n   Found pickle files: {pickle_files}")
                    # Try to load and examine
                    import pickle
                    with open(f'dataframe_folder/{pickle_files[0]}', 'rb') as f:
                        data = pickle.load(f)
                        print(f"   Pickle contains: {type(data).__name__}")
                        if hasattr(data, 'shape'):
                            print(f"   Shape: {data.shape}")
        else:
            print("❌ Output directory not created")
            
        # Check current directory for any outputs
        print("\n📂 Files in current directory:")
        for f in os.listdir('.'):
            if not f.startswith('.'):
                print(f"   - {f}")
                
    except Exception as e:
        print(f"\n❌ Error during execution: {e}")
        print("\nFull traceback:")
        traceback.print_exc()
        
        # Additional debug info
        print("\n🔍 Debug Information:")
        print(f"   Current directory: {os.getcwd()}")
        print(f"   Files in current directory: {[f for f in os.listdir('.') if not f.startswith('.')]}")
        if os.path.exists('csv_input_dir'):
            print(f"   Files in csv_input_dir: {os.listdir('csv_input_dir')}")
        if os.path.exists('dataframe_folder'):
            print(f"   Files in dataframe_folder: {os.listdir('dataframe_folder')}")

# Run the test
test_template_execution()

# ## 7. Alternative: Direct Function Test
# 
# If the above doesn't work, try calling the function directly

def direct_function_test():
    """Test by calling the function directly with Python objects"""
    
    print("\n=== Direct Function Test ===")
    
    try:
        # Import the template
        if not os.path.exists(template_path):
            print(f"❌ Template not found")
            return
            
        import importlib.util
        spec = importlib.util.spec_from_file_location("template_mod", template_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        
        # Try to call the function with direct parameters
        if hasattr(mod, 'LoadCSVFilesWithConfig'):
            print("Found LoadCSVFilesWithConfig function")
            
            # Create instance or call function
            # This depends on how your template is structured
            # Adjust as needed based on your actual template
            
        elif hasattr(mod, 'run_from_json'):
            # Create a minimal params file
            minimal_params = {
                "CSV_Files": os.path.abspath("csv_input_dir"),
                "CSV_Files_Configuration": os.path.abspath("config_table.csv"),
                "String_Columns": []  # Start with empty to see if that's the issue
            }
            
            with open('minimal_params.json', 'w') as f:
                json.dump(minimal_params, f, indent=2)
            
            print("Testing with minimal parameters:")
            print(json.dumps(minimal_params, indent=2))
            
            result = mod.run_from_json('minimal_params.json')
            print(f"✓ Function executed with minimal params")
            
    except Exception as e:
        print(f"❌ Error: {e}")
        traceback.print_exc()

# Uncomment to run this test
# direct_function_test()

# ## 8. Debugging the Galaxy Error
# 
# Based on your Galaxy error, let's check what might be wrong

print("\n=== Debugging Galaxy Integration ===")

# Check the wrapper script
wrapper_script = '../run_spac_template.sh'
if os.path.exists(wrapper_script):
    print("✓ Wrapper script found")
    # You can read and display key parts if needed
else:
    print("⚠️  Wrapper script not found at expected location")

# Simulate what Galaxy would pass
galaxy_params = {
    "CSV_Files": "csv_input_dir",  # Galaxy would pass a directory path
    "CSV_Files_Configuration": "config_table.csv",  # Galaxy would pass file path
    "String_Columns": "broad_cell_type,slide_number"  # Galaxy might pass as string
}

print("\nGalaxy-style parameters:")
print(json.dumps(galaxy_params, indent=2))

# The wrapper script should handle conversion
# Let's test String_Columns parsing
def test_string_columns_parsing():
    """Test different String_Columns formats"""
    
    test_cases = [
        "",  # Empty
        "[]",  # Empty list as string
        "column1",  # Single column
        "column1,column2",  # Comma-separated
        '["column1","column2"]',  # List as string
        ["column1", "column2"],  # Actual list
    ]
    
    print("\n=== Testing String_Columns Formats ===")
    for test_value in test_cases:
        print(f"\nInput: {repr(test_value)}")
        
        # Simulate parsing logic
        if isinstance(test_value, list):
            result = test_value
        elif test_value in ['', '[]', '[""]']:
            result = []
        elif test_value.startswith('['):
            try:
                import ast
                result = ast.literal_eval(test_value)
            except:
                result = []
        elif ',' in test_value:
            result = [s.strip() for s in test_value.split(',') if s.strip()]
        elif test_value:
            result = [test_value]
        else:
            result = []
            
        print(f"Output: {result} (type: {type(result).__name__})")

test_string_columns_parsing()

print("\n" + "="*50)
print("Testing complete! Check the output above for any errors.")
print("\nIf the template execution failed, common issues include:")
print("1. String_Columns not being a list")
print("2. File paths not being absolute or correct")
print("3. Missing output directory")
print("4. Template expecting different parameter names")

Working directory: /Users/liuf9/Projects/SCSAWorkflow/examples/spac_load_csv/test_run
Created test CSV files:
['sample_data_1.csv', 'sample_data_2.csv']

Configuration table:
           file_name slide_number patient_id timepoint
0  sample_data_1.csv    Slide_001  Patient_A  baseline
1  sample_data_2.csv    Slide_002  Patient_A     week4

⚠️  Please copy your template file to: /Users/liuf9/Projects/SCSAWorkflow/examples/spac_load_csv/test_run/load_csv_files_with_config_template.py
Once copied, continue with the next cell...
Parameters JSON created (main version):
{
  "CSV_Files": "csv_input_dir",
  "CSV_Files_Configuration": "config_table.csv",
  "Output_File": "dataframe_folder/combined_data.csv",
  "String_Columns": [
    "broad_cell_type",
    "slide_number",
    "patient_id",
    "timepoint"
  ]
}

Alternative parameters (if first doesn't work):
{
  "CSV_Files": "csv_input_dir",
  "CSV_Files_Configuration": "config_table.csv",
  "DataFrames": "dataframe_folder",
  "Output_File": "c